In [0]:
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as f
from torch.utils.data import IterableDataset, DataLoader
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, StructType, StructField, ArrayType, StringType
from pyspark.sql.functions import col, lit
from pyspark.ml.feature import StringIndexer
import pyarrow.parquet as pq
from itertools import chain
import pickle
import json

In [0]:
# ============================================================
# Load and transform data using fitted pipelines
# ============================================================

def load_spark_dataframes(spark, db, table_names, run_date):
    """Load inference dataframes for a specific date."""
    user_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['user_processed_features']}
        WHERE partition_date = '{run_date}'
    """)
    
    post_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['post_processed_features']}
        WHERE partition_date = '{run_date}'
    """)
    
    # Cache since we'll use them multiple times
    user_df.cache()
    post_df.cache()
    
    return user_df, post_df


def load_metadata(metadata_dir):
    """
    Load fitted metadata.

    """
    # Load known IDs 
    known_user_ids = pickle.load(open(Path(metadata_dir) / 'known_user_ids.pkl', 'rb'))
    known_post_ids = pickle.load(open(Path(metadata_dir) / 'known_post_ids.pkl', 'rb'))
    
    # Load categorical values
    known_user_cat_values = pickle.load(open(Path(metadata_dir) / 'known_user_cat_values.pkl', 'rb'))
    known_post_cat_values = pickle.load(open(Path(metadata_dir) / 'known_post_cat_values.pkl', 'rb'))
    
    return {
        "known_user_ids": known_user_ids,
        "known_post_ids": known_post_ids,
        "known_user_cat_values": known_user_cat_values,
        "known_post_cat_values": known_post_cat_values
    }


def process_inference_dataframe(df, id_col, cat_cols, num_cols, emb_col):
    """
    Process inference data using fitted pipeline.
    
    Args:
        df: Input dataframe
        id_col: ID column name
        cat_cols: List of categorical column names
        num_cols: List of numerical column names
        emb_col: Embedding column name
    
    Returns:
        Processed dataframe and ID decoder mapping
    """
    df_proc = df
    # Create temporary integer encoding for IDs (for tensor compatibility)
    # This is different from the model input encoding - it's just for indexing
    
    indexer = StringIndexer(
        inputCol=id_col,
        outputCol=id_col + "_temp_idx",
        handleInvalid="keep"   # avoid failures if new IDs appear
    )

    # Fit the indexer (distributed operation)
    model = indexer.fit(df)
    # Transform the dataframe
    df_proc = model.transform(df)
    
    # Extract decoder mapping: index → original ID
    # labelsArray is an ordered list: index -> string
    labels = model.labels  # already collected efficiently
    decode_id_map = {i: lbl for i, lbl in enumerate(labels)}

    # Select final columns
    select_cols = [
        id_col,
        id_col + '_temp_idx',
        id_col + "_idx",
        *[f"{c}_idx" for c in cat_cols],
        *[f"{c}_norm" for c in num_cols],
        emb_col
    ]
    
    df_proc = df_proc.select(*select_cols)
    
    return df_proc, decode_id_map


def process_user_post_inference(user_df, post_df,
                                user_id_col, post_id_col,
                                user_cat_cols, user_num_cols,
                                post_cat_cols, post_num_cols,
                                user_emb_col, post_emb_col,
                                parquet_dir):
    """
    Process user and post data for inference
    """
    
    # Process User Table
    user_df_proc, decode_uid_map = process_inference_dataframe(
        user_df,
        user_id_col,
        user_cat_cols,
        user_num_cols,
        user_emb_col
    )
    print("user table process completed")

    # Process Post Table
    post_df_proc, decode_pid_map = process_inference_dataframe(
        post_df,
        post_id_col,
        post_cat_cols,
        post_num_cols,
        post_emb_col
    )
    print("post table process completed")

    # Get statistics (useful for validation)
    n_users = user_df_proc.select(F.countDistinct(user_id_col + "_temp_idx")).first()[0]
    n_posts = post_df_proc.select(F.countDistinct(post_id_col + "_temp_idx")).first()[0]
    
    print("start writing tables...")

    # Write processed data
    user_dir = f"{parquet_dir}/users"
    post_dir = f"{parquet_dir}/posts"
    
    user_df_proc.repartition(100).write.mode("overwrite").option("compression", "snappy").parquet(user_dir)
    post_df_proc.repartition(100).write.mode("overwrite").option("compression", "snappy").parquet(post_dir)
    
    print("Writing completed")

    # Unpersist cached dataframes
    user_df.unpersist()
    post_df.unpersist()
    
    return {
        'n_users': n_users,
        'n_posts': n_posts,
        'user_dir': user_dir,
        'post_dir': post_dir,
        'uid_decoder': decode_uid_map,
        'pid_decoder': decode_pid_map
    }

In [0]:
def select_active_users(spark, run_date, lookback_win, db, table_names, to_pandas=False):
    sdf = spark.sql("""
                    select distinct user_id 
                    from {db}.{user_attributes}
                    where community_visits > 0 
                    and partition_date between date_sub('{run_date}', {lookback_win}) and '{run_date}'
                    """.format(db = db,
                               user_attributes = table_names["user_attributes"],
                               run_date = run_date,
                               lookback_win = lookback_win)
    )
    return sdf.toPandas() if to_pandas else sdf

def filter_active_users(df, user_set_df):
    # keeps rows in df where user_id exists in user_set_df
    filtered_df = df.join(user_set_df.select("user_id"), on="user_id", how="left_semi")
    return filtered_df 

In [0]:
# ============================================================
# Parquet Streaming Dataset
# ============================================================
class ParquetTwoTowerDataset(IterableDataset):
    def __init__(self, user_parquet_dir, post_parquet_dir,
                 user_id_int_col, post_id_int_col,
                 user_id_col, post_id_col,
                 user_cat_cols, user_num_cols,
                 post_cat_cols, post_num_cols,
                 user_emb_col, post_emb_col,
                 batch_size=1024):
        super().__init__()
        # Keep the raw IDs so that unknown IDs that are mapped to 0 can be traced back
        self.user_id_int_col, self.post_id_int_col = user_id_int_col, post_id_int_col
        self.user_id_col, self.post_id_col = user_id_col, post_id_col 
        self.user_cat_cols, self.user_num_cols = user_cat_cols, user_num_cols
        self.post_cat_cols, self.post_num_cols = post_cat_cols, post_num_cols
        self.user_emb_col, self.post_emb_col = user_emb_col, post_emb_col
        self.batch_size = batch_size

        self.user_files = sorted(Path(user_parquet_dir).glob("*.parquet"))
        self.post_files = sorted(Path(post_parquet_dir).glob("*.parquet"))

    # Validation for empty batches
    def _load_data(self, files):
        for file in files:
            table = pq.read_table(file)
            if table.num_rows == 0:
                continue  # Skip empty files
            for i in range(0, table.num_rows, self.batch_size):
                batch_size = min(self.batch_size, table.num_rows - i)
                if batch_size <= 0:
                    continue  # Skip empty batches
                batch = table.slice(i, batch_size)
                yield batch

    def _process_features(self, batch, id_col, id_int_col, cat_cols, num_cols, emb_col, default_emb_dim=512):
        # Process id
        id_int = torch.tensor(batch[id_int_col].to_numpy(), dtype=torch.long)
        id = torch.tensor(batch[id_col].to_numpy(), dtype=torch.long)
        # Process categoricals
        cat = (
            torch.as_tensor(
                np.column_stack([batch[col].to_numpy() for col in cat_cols]).astype(np.int64, copy=False),
                dtype=torch.long,
            )
            if cat_cols
            else torch.empty((id.size(0), 0), dtype=torch.long)
        )
        # Process numericals
        num = (
            torch.as_tensor(
                np.column_stack([batch[col].to_numpy() for col in num_cols]).astype(np.float32, copy=False),
                dtype=torch.float32,
            )
            if num_cols
            else torch.empty((id.size(0), 0), dtype=torch.float32)
        )
        
        # Process embeddings with shape validation
        emb_arrays = batch[emb_col].to_numpy()
        # Ensure all embeddings have the same shape
        emb_dim = len(emb_arrays[0]) if len(emb_arrays) > 0 else default_emb_dim
        padded_embs = []
        for emb in emb_arrays:
            emb = np.asarray(emb, dtype=np.float32)
            if len(emb) != emb_dim:
                # Pad or truncate to match expected dimension
                if len(emb) > emb_dim:
                    emb = emb[:emb_dim]
                else:
                    emb = np.pad(emb, (0, emb_dim - len(emb)), mode='constant')
            padded_embs.append(emb)
        emb = torch.tensor(np.stack(padded_embs), dtype=torch.float32)

        return {'id_int': id_int, 'id':id, "cat":cat, "num":num, "emb":emb}

    def _with_batch_type(self, batch_tensors, key_mapping, batch_type):
        mapped = {key_mapping.get(k, k): v for k, v in batch_tensors.items()}
        mapped["batch_type"] = batch_type
        return mapped

    def __iter__(self):
        user_key_mapping = {'id_int':'user_id_int','id':'user_id','cat':'user_cat','num':'user_num','emb':'lastn_embs'}
        for user_batch in self._load_data(self.user_files):
            user_batch_tensors = self._process_features(user_batch, self.user_id_col, self.user_id_int_col, self.user_cat_cols, self.user_num_cols, self.user_emb_col)
            yield self._with_batch_type(user_batch_tensors, user_key_mapping, "user")

        post_key_mapping = {'id_int':'post_id_int','id':'post_id','cat':'post_cat','num':'post_num','emb':'post_emb'}
        for post_batch in self._load_data(self.post_files):
            post_batch_tensors = self._process_features(post_batch, self.post_id_col, self.post_id_int_col, self.post_cat_cols, self.post_num_cols, self.post_emb_col)
            yield self._with_batch_type(post_batch_tensors, post_key_mapping, "post")

In [0]:
# ============================================================
# Model
# ============================================================
class MLP(nn.Module):
    """
    Multi-Layer Perceptron (MLP) used to process concatenated feature vectors.
    Consists of several linear layers with ReLU activations and dropout for regularization.
    """
    def __init__(self, in_dim, hidden_dims=(256,128), dropout=0.2):
        super().__init__()
        layers = []  # List to hold layers
        prev = in_dim  # Track input dimension for each layer
        for h in hidden_dims:
            # Add a Linear layer, followed by ReLU and Dropout
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h  # Update input dimension for next layer
        self.net = nn.Sequential(*layers)  # Combine layers into a sequential model

    def forward(self, x):
        # Forward pass through the MLP
        return self.net(x)

class UserTower(nn.Module):
    """
    Encodes user features into a dense vector representation.
    Combines user ID, categorical features, numerical features, and sequence embeddings (e.g., user history).
    """
    def __init__(self, uid_vocab, uid_dim, user_cat_dims, user_num_dim, lastn_emb_dim, hidden_dims, dropout):
        super().__init__()
        # Embedding for user ID (with padding index 0). Add +1 to vocab size for unknown IDs
        self.user_id_emb = nn.Embedding(uid_vocab + 1, uid_dim, padding_idx=0)
        # Store the known vocab size for later use
        self.known_uid_vocab = uid_vocab   
        # Embeddings for each categorical user feature
        self.user_cat_embs = nn.ModuleList([
            nn.Embedding(v + 1, d, padding_idx=0) for v, d in user_cat_dims
        ])
        self.known_user_cat_dims = user_cat_dims   
        # MLP to combine all user features into a single vector
        self.mlp = MLP(uid_dim + sum(d for _, d in user_cat_dims) + user_num_dim + lastn_emb_dim, 
                      hidden_dims, dropout)

    def forward(self, user_id, user_cat, user_num, lastn_embs):
        # Map unknown user IDs to the special unknown index (0)
        user_id = torch.where(user_id >= self.known_uid_vocab, 
                             torch.zeros_like(user_id), 
                             user_id)
        
        u_id = self.user_id_emb(user_id)
        
        # Embed and concatenate all categorical features (if any)
        if self.user_cat_embs:
            processed_cats = []
            for i, (emb, (v, _)) in enumerate(zip(self.user_cat_embs, self.known_user_cat_dims)):
                # Map unknown category values to 0
                cat_col = user_cat[:, i]
                cat_col = torch.where(cat_col >= v, torch.zeros_like(cat_col), cat_col)
                processed_cats.append(emb(cat_col))
            u_cat = torch.cat(processed_cats, dim=-1)
        else:
            # If no categorical features, create empty tensor
            u_cat = torch.zeros(u_id.size(0), 0, device=u_id.device)

        # Concatenate all features and pass through MLP  
        return self.mlp(torch.cat([u_id, u_cat, user_num, lastn_embs], dim=-1))

class ItemTower(nn.Module):
    """
    Encodes post/item features into a dense vector representation.
    Combines post ID, categorical features, numerical features, and content embeddings.
    """
    def __init__(self, pid_vocab, pid_dim, post_cat_dims, post_num_dim, post_emb_dim, hidden_dims, dropout):
        super().__init__()
        # Embedding for post ID (with padding index 0). Add +1 to vocab size for unknown IDs
        self.post_id_emb = nn.Embedding(pid_vocab + 1, pid_dim, padding_idx=0)
        self.known_pid_vocab = pid_vocab
        # Embeddings for each categorical post feature
        self.post_cat_embs = nn.ModuleList([
            nn.Embedding(v + 1, d, padding_idx=0) for v, d in post_cat_dims
        ])
        self.known_post_cat_dims = post_cat_dims
        
        self.mlp = MLP(pid_dim + sum(d for _, d in post_cat_dims) + post_num_dim + post_emb_dim, 
                      hidden_dims, dropout)

    def forward(self, post_id, post_cat, post_num, post_emb):
        # Map unknown post IDs to the special unknown index (0)
        post_id = torch.where(post_id >= self.known_pid_vocab, 
                             torch.zeros_like(post_id), 
                             post_id)
        
        p_id = self.post_id_emb(post_id)
        
        # Embed and concatenate all categorical features (if any)
        if self.post_cat_embs:
            processed_cats = []
            for i, (emb, (v, _)) in enumerate(zip(self.post_cat_embs, self.known_post_cat_dims)):
                # Map unknown category values to 0
                cat_col = post_cat[:, i]
                cat_col = torch.where(cat_col >= v, torch.zeros_like(cat_col), cat_col)
                processed_cats.append(emb(cat_col))
            p_cat = torch.cat(processed_cats, dim=-1)
        else:
            # If no categorical features, create empty tensor
            p_cat = torch.zeros(p_id.size(0), 0, device=p_id.device)
        # Concatenate all features and pass through MLP    
        return self.mlp(torch.cat([p_id, p_cat, post_num, post_emb], dim=-1))

def _last_linear_out_features(seq):
    """
    Utility to get output dimension of last Linear layer in a Sequential.
    """
    for l in reversed(seq):
        if isinstance(l, nn.Linear): return l.out_features
    raise ValueError("No Linear found")

class TwoTowerModel(nn.Module):
    """
    Combines user and item towers, projects them to a common space,
    and computes similarity scores for retrieval.
    """
    def __init__(self, user_tower, item_tower, proj_dim, normalize=True, temperature=0.1):
        super().__init__()
        self.user_tower, self.item_tower = user_tower, item_tower
        self.user_proj = nn.Linear(_last_linear_out_features(user_tower.mlp.net), proj_dim)
        self.item_proj = nn.Linear(_last_linear_out_features(item_tower.mlp.net), proj_dim)
        self.normalize = normalize
        self.temperature = temperature
    def encode_user(self,*a):
        z = self.user_proj(self.user_tower(*a)); return f.normalize(z,dim=-1) if self.normalize else z
    def encode_item(self,*a):
        z = self.item_proj(self.item_tower(*a)); return f.normalize(z,dim=-1) if self.normalize else z
    def forward(self, user_inputs, item_inputs):
        # Computes similarity scores between user and item vectors
        u = self.encode_user(*user_inputs); v = self.encode_item(*item_inputs)
        return (u @ v.t())/self.temperature

In [0]:
# ============================================================
# Model Inference
# ============================================================
def model_inference_single_user(model, data_loader, device="cpu", top_k=100):
    """Process one user at a time for maximum memory efficiency"""
    model.eval()
    
    # Collect all post embeddings first
    post_embeddings = []
    post_ids = []
    
    with torch.no_grad():
        for batch in data_loader:
            if batch["batch_type"] == "post":
                # Process post batch
                item_inputs = (
                    batch['post_id'].to(device),
                    batch['post_cat'].to(device),
                    batch['post_num'].to(device),
                    batch['post_emb'].to(device)
                )
                post_emb = model.encode_item(*item_inputs)
                post_embeddings.append(post_emb.cpu())
                post_ids.append(batch['post_id_int'].cpu())
    
    # Concatenate all post embeddings
    if post_embeddings:
        post_embeddings = torch.cat(post_embeddings, dim=0)
        post_ids = torch.cat(post_ids, dim=0)
    else:
        post_embeddings = torch.empty((0, model.item_proj.out_features))
        post_ids = torch.empty((0,), dtype=torch.long)
    
    # Process users one by one
    user_topk_scores = []
    user_topk_indices = []
    user_ids_list = []
    
    with torch.no_grad():
        for batch in data_loader:
            if batch["batch_type"] == "user":
                # Process user batch
                user_inputs = (
                    batch['user_id'].to(device),
                    batch['user_cat'].to(device),
                    batch['user_num'].to(device),
                    batch['lastn_embs'].to(device)
                )
                user_emb = model.encode_user(*user_inputs)
                user_id = batch['user_id_int'].cpu()
                
                # Compute similarity with all posts
                similarity = user_emb @ post_embeddings.t()
                
                # Get top-K
                topk_scores, topk_indices = torch.topk(similarity.squeeze(), k=min(top_k, similarity.size(1)))
                
                user_topk_scores.append(topk_scores)
                user_topk_indices.append(topk_indices)
                user_ids_list.append(user_id)
                
                # Clear memory
                del similarity, topk_scores, topk_indices
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    
    # Combine results
    if user_topk_scores:
        topk_scores = torch.stack(user_topk_scores, dim=0)
        topk_indices = torch.stack(user_topk_indices, dim=0)
        user_ids = torch.cat(user_ids_list, dim=0)
    else:
        topk_scores = torch.empty((0, top_k))
        topk_indices = torch.empty((0, top_k), dtype=torch.long)
        user_ids = torch.empty((0,), dtype=torch.long)
    
    return {
        "topk_scores": topk_scores,
        "topk_indices": topk_indices,
        "user_ids": user_ids,
        "post_ids": post_ids
    }

def model_inference_streaming(model, data_loader, device="cpu", user_chunk_size=100, post_chunk_size=1000):
    """Memory-efficient evaluation function for streaming datasets"""
    model.eval()
    
    # Separate user and item embeddings
    user_embeddings = []
    post_embeddings = []
    user_ids = []
    post_ids = []
    
    with torch.no_grad():
        for batch in data_loader:
            if batch["batch_type"] == "user":
                # Process user batch
                user_inputs = (
                    batch['user_id'].to(device),
                    batch['user_cat'].to(device),
                    batch['user_num'].to(device),
                    batch['lastn_embs'].to(device)
                )
                user_emb = model.encode_user(*user_inputs)
                user_embeddings.append(user_emb.cpu())
                user_ids.append(batch['user_id_int'].cpu())
                
            elif batch["batch_type"] == "post":
                # Process post batch
                item_inputs = (
                    batch['post_id'].to(device),
                    batch['post_cat'].to(device),
                    batch['post_num'].to(device),
                    batch['post_emb'].to(device)
                )
                post_emb = model.encode_item(*item_inputs)
                post_embeddings.append(post_emb.cpu())
                post_ids.append(batch['post_id_int'].cpu())
    
    # Concatenate all embeddings
    if user_embeddings:
        user_embeddings = torch.cat(user_embeddings, dim=0)
        user_ids = torch.cat(user_ids, dim=0)
    else:
        user_embeddings = torch.empty((0, model.user_proj.out_features))
        user_ids = torch.empty((0,), dtype=torch.long)
    
    if post_embeddings:
        post_embeddings = torch.cat(post_embeddings, dim=0)
        post_ids = torch.cat(post_ids, dim=0)
    else:
        post_embeddings = torch.empty((0, model.item_proj.out_features))
        post_ids = torch.empty((0,), dtype=torch.long)
    
    # If either users or posts are empty, return empty results
    if user_embeddings.size(0) == 0 or post_embeddings.size(0) == 0:
        return {
            "scores": torch.empty((0, 0)),
            "user_ids": user_ids,
            "post_ids": post_ids,
            "user_embeddings": user_embeddings,
            "post_embeddings": post_embeddings
        }
    
    # Compute similarity matrix in small chunks to avoid memory issues
    all_scores = []
    
    # Process users in chunks
    for i in range(0, user_embeddings.size(0), user_chunk_size):
        user_chunk = user_embeddings[i:i+user_chunk_size]
        chunk_scores = []
        
        # Process posts in chunks
        for j in range(0, post_embeddings.size(0), post_chunk_size):
            post_chunk = post_embeddings[j:j+post_chunk_size]
            
            # Compute similarity between user_chunk and post_chunk
            similarity = user_chunk @ post_chunk.t()
            chunk_scores.append(similarity)
            
            # Clear memory
            del similarity
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        # Combine post chunks for this user chunk
        if chunk_scores:
            chunk_result = torch.cat(chunk_scores, dim=1)
            all_scores.append(chunk_result)
            
            # Clear memory
            del chunk_scores, chunk_result
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # Combine all user chunks
    if all_scores:
        scores = torch.cat(all_scores, dim=0)
    else:
        scores = torch.empty((0, 0))
    
    return {
        "scores": scores,
        "user_ids": user_ids,
        "post_ids": post_ids,
        "user_embeddings": user_embeddings,
        "post_embeddings": post_embeddings
    }  

def model_inference_topk(model, data_loader, device="cpu", top_k=100, user_chunk_size=100, post_chunk_size=1000):
    """Memory-efficient top-K recommendation function"""
    model.eval()
    
    # Separate user and item embeddings
    user_embeddings = []
    post_embeddings = []
    user_ids = []
    post_ids = []
    
    with torch.no_grad():
        for batch in data_loader:
            if batch["batch_type"] == "user":
                # Process user batch
                user_inputs = (
                    batch['user_id'].to(device),
                    batch['user_cat'].to(device),
                    batch['user_num'].to(device),
                    batch['lastn_embs'].to(device)
                )
                user_emb = model.encode_user(*user_inputs)
                user_embeddings.append(user_emb.cpu())
                user_ids.append(batch['user_id_int'].cpu())
                
            elif batch["batch_type"] == "post":
                # Process post batch
                item_inputs = (
                    batch['post_id'].to(device),
                    batch['post_cat'].to(device),
                    batch['post_num'].to(device),
                    batch['post_emb'].to(device)
                )
                post_emb = model.encode_item(*item_inputs)
                post_embeddings.append(post_emb.cpu())
                post_ids.append(batch['post_id_int'].cpu())
    
    # Concatenate all embeddings
    if user_embeddings:
        user_embeddings = torch.cat(user_embeddings, dim=0)
        user_ids = torch.cat(user_ids, dim=0)
    else:
        return {
            "topk_scores": torch.empty((0, top_k)),
            "topk_indices": torch.empty((0, top_k), dtype=torch.long),
            "user_ids": torch.empty((0,), dtype=torch.long),
            "post_ids": torch.empty((0,), dtype=torch.long)
        }
    
    if post_embeddings:
        post_embeddings = torch.cat(post_embeddings, dim=0)
        post_ids = torch.cat(post_ids, dim=0)
    else:
        return {
            "topk_scores": torch.empty((0, top_k)),
            "topk_indices": torch.empty((0, top_k), dtype=torch.long),
            "user_ids": user_ids,
            "post_ids": torch.empty((0,), dtype=torch.long)
        }
    
    # Compute top-K recommendations without storing the full similarity matrix
    all_topk_scores = []
    all_topk_indices = []
    
    # Process users in chunks
    for i in range(0, user_embeddings.size(0), user_chunk_size):
        user_chunk = user_embeddings[i:i+user_chunk_size]
        chunk_topk_scores = torch.full((user_chunk.size(0), top_k), -float('inf'))
        chunk_topk_indices = torch.zeros((user_chunk.size(0), top_k), dtype=torch.long)
        
        # Process posts in chunks
        for j in range(0, post_embeddings.size(0), post_chunk_size):
            post_chunk = post_embeddings[j:j+post_chunk_size]
            post_indices = torch.arange(j, min(j+post_chunk_size, post_embeddings.size(0)))
            
            # Compute similarity between user_chunk and post_chunk
            similarity = user_chunk @ post_chunk.t()
            
            # Update top-K
            current_topk_scores, current_topk_indices = torch.topk(similarity, k=min(top_k, post_chunk.size(0)), dim=1)
            
            # Combine with previous top-K
            combined_scores = torch.cat([chunk_topk_scores, current_topk_scores], dim=1)
            combined_indices = torch.cat([chunk_topk_indices, post_indices[current_topk_indices]], dim=1)
            
            # Get new top-K
            chunk_topk_scores, topk_idx = torch.topk(combined_scores, k=top_k, dim=1)
            
            # Use advanced indexing to get the corresponding indices
            batch_indices = torch.arange(chunk_topk_scores.size(0)).unsqueeze(1).expand(-1, top_k)
            chunk_topk_indices = combined_indices[batch_indices, topk_idx]
            
            # Clear memory
            del similarity, current_topk_scores, current_topk_indices, combined_scores, combined_indices
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        
        all_topk_scores.append(chunk_topk_scores)
        all_topk_indices.append(chunk_topk_indices)
        
        # Clear memory
        del chunk_topk_scores, chunk_topk_indices
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Combine all user chunks
    if all_topk_scores:
        topk_scores = torch.cat(all_topk_scores, dim=0)
        topk_indices = torch.cat(all_topk_indices, dim=0)
    else:
        topk_scores = torch.empty((0, top_k))
        topk_indices = torch.empty((0, top_k), dtype=torch.long)
    
    return {
        "topk_scores": topk_scores,
        "topk_indices": topk_indices,
        "user_ids": user_ids,
        "post_ids": post_ids
    }

def optimized_model_inference(model, data_loader, device="cpu", strategy="topk", **kwargs):
    """Memory-optimized inference with multiple strategies"""
    
    # Strategy 1: Process one user at a time (slowest but most memory-efficient)
    if strategy == "single_user":
        return model_inference_single_user(model, data_loader, device, **kwargs)
    
    # Strategy 2: Process in chunks (balanced)
    elif strategy == "chunked":
        return model_inference_streaming(model, data_loader, device, **kwargs)
    
    # Strategy 3: Top-K only (most memory-efficient for recommendations)
    elif strategy == "topk":
        return model_inference_topk(model, data_loader, device, **kwargs)
    
    else:
        raise ValueError(f"Unknown strategy: {strategy}")


In [0]:
# ============================================================
# Write result
# ============================================================
def _decode_ids(encoded_ids, decoder, to_numpy=False):
    ids = [decoder[id] for id in encoded_ids]
    return np.array(ids) if to_numpy else ids

def retrieve_k_posts_per_user(spark, model_res, decoder_uid, decoder_pid, top_k, table_name, run_date, chunk_size=5000):
    uids = model_res['user_ids'].numpy()
    pids = model_res['post_ids'].numpy()
    top_ind = model_res['topk_indices'].numpy()
    top_ind = top_ind[:,:top_k]

    uids = _decode_ids(uids, decoder_uid)
    pids = _decode_ids(pids, decoder_pid, to_numpy=True)

    # define schema
    schema = StructType([
        StructField("user_id", StringType(), False),
        StructField("post_id", ArrayType(StringType()), False)
    ])

    # Process data in chunks to avoid memory issues
    for i in range(0, len(uids), chunk_size):
        chunk_data = []
        end_idx = min(i + chunk_size, len(uids))
        
        for j in range(i, end_idx):
            user_id = uids[j]
            indices = top_ind[j]
            posts = pids[indices].tolist()
            chunk_data.append((user_id, posts))
        
        chunk_df = spark.createDataFrame(chunk_data, schema=schema)
        chunk_df = chunk_df.withColumn("partition_date", lit(run_date))
        
        if i == 0:
            chunk_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
        else:
            chunk_df.write.mode("append").saveAsTable(table_name)

    return None

In [0]:
# ============================================================
# Orchestrator Helpers
# ============================================================
def build_model_from_dims(n_users, n_posts, user_cat_dims, post_cat_dims,
                          user_num_dim, post_num_dim,
                          id_emb_dim=16, cat_emb_dim=8, emb_dim=512, proj_dim=64,
                          hidden_dims=(256, 128), dropout=0.2, temperature=0.1):
    """Construct UserTower, ItemTower and wrap in TwoTowerModel."""
    user_tower = UserTower(
        uid_vocab=int(n_users), uid_dim=id_emb_dim,
        user_cat_dims=[(c, min(cat_emb_dim, c)) for c in user_cat_dims],
        user_num_dim=user_num_dim,
        lastn_emb_dim=emb_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    )
    item_tower = ItemTower(
        pid_vocab=int(n_posts), pid_dim=id_emb_dim,
        post_cat_dims=[(c, min(cat_emb_dim, c)) for c in post_cat_dims],
        post_num_dim=post_num_dim,
        post_emb_dim=emb_dim,
        hidden_dims=hidden_dims,
        dropout=dropout,
    )
    return TwoTowerModel(user_tower, item_tower, proj_dim=proj_dim, temperature=temperature)    

def init_data_loader(user_id_col, post_id_col, user_cat_cols, user_num_cols, post_cat_cols, post_num_cols, user_emb_col, post_emb_col, parquet_dir, batch_size=1024):
    dataset = ParquetTwoTowerDataset(
        user_parquet_dir=f"{parquet_dir}/users",
        post_parquet_dir=f"{parquet_dir}/posts",
        user_id_int_col=user_id_col+'_temp_idx',
        post_id_int_col=post_id_col+'_temp_idx',
        user_id_col=user_id_col+'_idx', 
        post_id_col=post_id_col+'_idx',
        user_cat_cols=[c+'_idx' for c in user_cat_cols],
        user_num_cols=[c+'_norm' for c in user_num_cols],
        post_cat_cols=[c+'_idx' for c in post_cat_cols],
        post_num_cols=[c+'_norm' for c in post_num_cols],
        user_emb_col=user_emb_col,
        post_emb_col=post_emb_col,
        batch_size=batch_size
    )

    data_loader = DataLoader(dataset, batch_size=None, num_workers=0)
    return data_loader

In [0]:
# ============================================================
# Orchestration
# ============================================================
def run_model_inference(data_args, model_args, chunk_args, metadata_dir, parquet_dir, model_path, top_k, active_user_win, db, table_names, run_date, batch_size=1024):

    print(f"Loading data for run_date: {run_date}")

    # Load inference data
    print("Loading inference data...")
    user_df, post_df = load_spark_dataframes(spark, db, table_names, run_date)

    user_set_df = select_active_users(spark, run_date, active_user_win, db, table_names)
    user_df = filter_active_users(user_df, user_set_df)

    # Load fitted pipelines and metadata
    print("Loading fitted model metadata...")
    metadata = load_metadata(metadata_dir)

    known_user_ids = metadata["known_user_ids"]
    known_post_ids = metadata["known_post_ids"]
    known_user_cat_values = metadata["known_user_cat_values"]
    known_post_cat_values = metadata["known_post_cat_values"]

    # Process data for inference
    print("Processing data for inference...")
    proc_res = process_user_post_inference(
        user_df, post_df,
        parquet_dir=parquet_dir,
        **data_args
    )

    # Print metadata
    print("\n" + "="*60)
    print("Data processing complete!")
    print("="*60)
    print(f"Users (inference):           {proc_res['n_users']}")
    print(f"Posts (inference):           {proc_res['n_posts']}")
    print(f"User data saved to:          {proc_res['user_dir']}")
    print(f"Post data saved to:          {proc_res['post_dir']}")
    print("="*60)

    print("Load model:")

    model = build_model_from_dims(n_users=len(known_user_ids), 
                                  n_posts=len(known_post_ids),
                                  user_cat_dims=[len(known_user_cat_values[c]) for c in data_args['user_cat_cols']],
                                  post_cat_dims=[len(known_post_cat_values[c]) for c in data_args['post_cat_cols']],
                                  user_num_dim=len(data_args['user_num_cols']),
                                  post_num_dim=len(data_args['post_num_cols']),
                                  **model_args)
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()
    print("Model structure:")
    print(model)
    
    # Initialize dataset and data loader
    data_loader = init_data_loader(parquet_dir=parquet_dir, batch_size=batch_size, **data_args)

    # Run inference
    print("Run inference:")
    results = optimized_model_inference(
        model, data_loader, device="cpu", 
        strategy="topk", top_k=top_k, 
        user_chunk_size=chunk_args["user_infer_chunk_size"], 
        post_chunk_size=chunk_args["post_infer_chunk_size"]
    )

    # write retrieval results to table
    print("Write results to table:")
    retrieve_k_posts_per_user(spark, 
                              results, 
                              proc_res['uid_decoder'], 
                              proc_res['pid_decoder'], 
                              top_k=top_k, 
                              table_name=db+'.'+table_names['two_tower_retrieval'], 
                              run_date=run_date,
                              chunk_size=chunk_args["table_write_chunk_size"])
    return None

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)

In [0]:
metadata_dir = model_config['TWO_TOWER_METADATA_PATH']
parquet_dir = model_config['TWO_TOWER_TRAIN_PARQUET_PATH']
model_path = model_config['TWO_TOWER_MODEL_PATH']

data_args = dict(
    user_id_col='user_id', 
    post_id_col='post_id',
    user_cat_cols=["ip_province", "vehicle_series", "platform"],
    user_num_cols=["months_since_registration", "months_since_consent", "identity", 
                    "is_koc", "has_profile_photo", "community_visits", "posts_published", 
                    "posts_viewed", "posts_liked", "posts_commented", "posts_shared", 
                    "users_followed", "tribes_joined"],
    post_cat_cols=["post_type"], 
    post_num_cols=["days_since_published", "is_ugc", "is_sink", "has_pic", "has_video",
                    "views", "likes", "comments", "shares", "dwell_time"],
    user_emb_col='lastn_embs', 
    post_emb_col='post_content_emb'
)

model_args = model_config['model_args']

top_k = model_config['top_k_posts']
chunk_args = model_config['chunk_args']
batch_size = model_config['batch_size']
active_user_win = model_config.get('active_user_window', model_config.get('active_user_win', 30))

In [0]:
# ============================================================
# Run inference pipeline
# ============================================================
run_model_inference(data_args, model_args, chunk_args, metadata_dir, parquet_dir, model_path, top_k, active_user_win, db, table_names, run_date, batch_size)